# 04 — Vanilla RNN with Flax NNX

We build an Elman RNN using **`flax.nnx`**, the newer Flax API that:
- Uses standard Python classes (no `@nn.compact` magic)
- Holds parameters as mutable module attributes
- Works with `nnx.jit`, `nnx.grad`, `nnx.Optimizer` — NNX-aware versions of JAX transforms

For sequential computation, we use `jax.lax.scan` with the `nnx.split / nnx.merge` pattern to remain fully JIT-compilable.

In [ ]:
import sys; sys.path.insert(0, '..')
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import optax
import numpy as np
import matplotlib.pyplot as plt

from src.rnn_jax import VanillaRNNCell, VanillaRNN, VanillaRNNModel
from src.utils import make_sinusoid_task, plot_loss_curves, plot_predictions
from src import train as T

jax.config.update('jax_enable_x64', False)   # float32 for NN training

## 4.1 The NNX Module System

An `nnx.Module` is a Python class. Parameters are stored as `nnx.Param` attributes (or inside sub-modules like `nnx.Linear`). There is no `init` / `apply` call — you just instantiate and call.

In [ ]:
# Inspect the module structure
rngs = nnx.Rngs(params=0)   # seed for parameter initialisation
cell = VanillaRNNCell(input_size=3, hidden_size=8, rngs=rngs)

# Parameters are explicit nnx.Param attributes
print('W shape:   ', cell.W.shape)     # (hidden, hidden)  — recurrent
print('W_in shape:', cell.W_in.shape)  # (hidden, input)   — input
print('b shape:   ', cell.b.shape)     # (hidden,)         — bias
print('alpha:     ', cell.alpha)            # leaky time constant

# Forward pass
h = jnp.zeros((1, 8))   # (batch, hidden)
x = jnp.ones((1, 3))    # (batch, input)
h_new = cell(h, x)
print('h_new shape:', h_new.shape)

In [ ]:
# nnx.split: separate the module into static graph + dynamic state pytree
graphdef, state = nnx.split(cell)
leaves = jax.tree.leaves(state)
print(f'Extracted state: {len(leaves)} arrays, {sum(x.size for x in leaves)} total params')

# nnx.merge: reconstruct the module from graphdef + state
cell_copy = nnx.merge(graphdef, state)
h_check = cell_copy(h, x)
print('Outputs match:', jnp.allclose(h_new, h_check))

## 4.2 VanillaRNN: Full-Sequence Forward Pass

The `VanillaRNN` module runs a sequence via `lax.scan`. The `nnx.split` / `nnx.merge` trick makes the step function pure (required by scan) while keeping params accessible.

In [ ]:
INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE = 1, 32, 1
T_LEN, BATCH = 50, 16

model = VanillaRNNModel(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE,
    output_size=OUTPUT_SIZE,
    rngs=nnx.Rngs(0),
)

xs = jnp.ones((BATCH, T_LEN, INPUT_SIZE))   # batch-first: (B, T, I)
preds = model(xs)
print('Input shape: ', xs.shape)
print('Output shape:', preds.shape)   # (B, T, output_size)

In [ ]:
# Count parameters
_, state = nnx.split(model)
n_params = sum(v.size for v in jax.tree.leaves(state))
print(f'Total parameters: {n_params:,}')

## 4.3 Training: Sinusoid Prediction Task

The network receives `sin(2π f t)` and must predict the next value `sin(2π f (t+1))`. Each sequence has a randomly drawn frequency.

In [ ]:
N_SEQ, SEQ_LEN = 512, 100
data = make_sinusoid_task(N_SEQ, SEQ_LEN, key=jax.random.PRNGKey(0))
print('inputs  shape:', data['inputs'].shape)   # (N, T, 1) — batch-first
print('targets shape:', data['targets'].shape)

In [ ]:
model = VanillaRNNModel(
    input_size=1, hidden_size=64, output_size=1, rngs=nnx.Rngs(42)
)

def loss_fn(model, batch):
    preds = model(batch['inputs'])
    return T.mse_loss(preds, batch['targets'])

losses = T.fit(
    model, data,
    loss_fn=loss_fn,
    n_steps=500,
    lr=3e-3,
    batch_size=64,
    log_every=100,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_loss_curves(losses, title='VanillaRNN training loss', ax=axes[0])

# Predictions on a held-out batch
test_data = make_sinusoid_task(8, SEQ_LEN, key=jax.random.PRNGKey(999))
test_preds = model(test_data['inputs'])

t = np.arange(SEQ_LEN)
axes[1].plot(t, np.array(test_data['targets'][0, :, 0]), label='target', lw=2)
axes[1].plot(t, np.array(test_preds[0, :, 0]), '--', label='predicted', lw=2)
axes[1].set(xlabel='t', title='Example prediction (test set)')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4.4 Gradient Flow: Exploding / Vanishing Gradients

Vanilla RNNs struggle with long sequences because gradients either explode or vanish as they backpropagate through T steps. Let's visualise the gradient norm as a function of sequence position.

In [ ]:
def grad_norm_per_step(model, xs, targets):
    """Compute ‖∂L/∂h_0‖ when h_0 is inserted at each t_start (proxy for vanishing grads)."""
    B, T_len, _ = xs.shape
    graphdef, state = nnx.split(model.rnn.cell)

    def loss_from_h0(h0, t_start):
        # h0: (B, H) — differentiated w.r.t.
        # t_start: Python int constant, not differentiated
        def run_single(x_seq, h_init):
            # x_seq: (T_rem, I), h_init: (H,) — one sequence, one h_init
            def step(h, x):
                cell = nnx.merge(graphdef, state)
                return cell(h, x), cell(h, x)
            h_final, _ = jax.lax.scan(step, h_init, x_seq)
            return h_final

        h_finals = jax.vmap(run_single)(xs[:, t_start:], h0)  # (B, H)
        preds = model.readout(h_finals)                         # (B, O)
        return jnp.mean((preds - targets[:, -1]) ** 2)         # targets[:, -1]: last timestep

    h0_zeros = jnp.zeros((B, model.rnn.hidden_size))
    grad_norms = []
    for t_start in range(0, T_len, 10):
        g = jax.grad(loss_from_h0)(h0_zeros, t_start)  # ∂L/∂h0, shape (B, H)
        grad_norms.append(float(jnp.mean(jnp.linalg.norm(g, axis=-1))))
    return grad_norms

# Quick demo with short sequence to avoid slow compile
demo_data = make_sinusoid_task(16, 50, key=jax.random.PRNGKey(7))
model_demo = VanillaRNNModel(1, 32, 1, rngs=nnx.Rngs(0))
gnorms = grad_norm_per_step(model_demo, demo_data['inputs'], demo_data['targets'])

plt.figure(figsize=(8, 3))
plt.semilogy(range(0, 50, 10), gnorms, 'o-')
plt.xlabel('Steps from end'); plt.ylabel('Mean gradient norm')
plt.title('Gradient norm as function of distance from loss')
plt.gca().invert_xaxis()   # left = near loss, right = far
plt.grid(alpha=0.3); plt.show()